In [ ]:
# nbplay runs entirely in your browser: install the bundled wheel first.
%pip install -q nbplay


# 04 SamplerWidget Tutorial

Load a generated sample and inspect ADSR behavior.

In [ ]:
import math

from nbplay import SamplerWidget

sampler = SamplerWidget(
    attack=0.005,
    decay=0.08,
    sustain=0.15,
    release=0.12,
    max_voices=6,
)

sample_rate = 44100
duration_seconds = 0.18
frame_count = int(sample_rate * duration_seconds)
sample_data = [
    math.sin(2 * math.pi * 880 * index / sample_rate) * (1 - index / frame_count)
    for index in range(frame_count)
]

sampler.load_sample(sample_data, sample_rate=sample_rate, root_note=60, name="Notebook Blip")
sampler

## Zones: One Sample per Key Range

A sampler can hold more than one sample. Each zone maps a note range (and velocity range) to its own audio and root note; a note that lands in a zone plays that zone instead of the main sample. Load zones from Python, from a file in the widget, or record the microphone straight onto a pad with the Rec button.

In [ ]:
kick = [math.sin(2 * math.pi * 60 * i / sample_rate) * (1 - i / 4000) for i in range(4000)]
snare = [(1 - i / 3000) * ((i * 7919) % 200 / 100 - 1) for i in range(3000)]

zoned = SamplerWidget(pad_notes=[36, 38, 42, 46])
zoned.add_zone(kick, 36, 36, sample_rate=sample_rate, name="Kick")
zoned.add_zone(snare, 38, 38, sample_rate=sample_rate, name="Snare")
zoned.add_zone(sample_data, 48, 72, root_note=60, sample_rate=sample_rate, name="Blip keys")
print(zoned.zone_for(60)["name"], "|", len(zoned.to_sample_map()), "mappings")
zoned

## Take to Pad

A take recorded on the timeline can become a pad. `capture_clip()` hands the clip to the sampler in the browser, which decodes it (honouring the clip's trim) and adds a zone on the pad's note. Both widgets must be displayed in the page.

In [ ]:
from nbplay import Session

capture_session = Session()
capture_session.add_track("Mic", armed=True)
drums = capture_session.add_track("Drums", sound_source=zoned)
# After recording a take on the Mic lane:
# capture_session.timeline.clips[0] -> zoned.capture_clip(capture_session.timeline, clip["id"], pad=2)
capture_session.timeline